In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import re
import sys

from partd import python
from pathlib import Path

sys.path.append(os.path.abspath('../../'))
from utils_mitgcm import *
from utils_modal_analysis import _match_mode_to_field, project_vector_mode

# Load MITgcm results

In [2]:
lake = 'zug'
model = f'{lake}_2025'

In [3]:
mitgcm_config, ds_mitgcm = open_mitgcm_ds_from_config('../../config.json', model)
base_folder_path = os.path.dirname(mitgcm_config['datapath'])

In [4]:
horizontal_resolution = 100
ds_mitgcm['YG'] = np.arange(0, len(ds_mitgcm['YG'])) * horizontal_resolution
ds_mitgcm['XG'] = np.arange(0, len(ds_mitgcm['XG'])) * horizontal_resolution
ds_mitgcm['YC'] = np.arange(1, len(ds_mitgcm['YC']) + 1) * horizontal_resolution - horizontal_resolution / 2
ds_mitgcm['XC'] = np.arange(1, len(ds_mitgcm['XC']) + 1) * horizontal_resolution - horizontal_resolution / 2

In [5]:
ds_mitgcm['UVEL']=ds_mitgcm.UVEL.where(ds_mitgcm.UVEL != 0, np.nan)
ds_mitgcm['VVEL']=ds_mitgcm.VVEL.where(ds_mitgcm.VVEL != 0, np.nan)

# Get folder & files

In [6]:
base_folder = rf"/storage/alplakes_test/{lake}_100m_2025"

In [7]:
modal_analysis_dir = os.path.join(base_folder_path, "modal_analysis")

In [8]:
mode_date_str = "2025-08-07"
date_dir = Path(modal_analysis_dir) / mode_date_str
files = list(date_dir.rglob("*mode*.nc"))

# Get layers

In [9]:
stratification = pd.read_csv(os.path.join(modal_analysis_dir, 'mean_profiles_2025_bimonthly.csv')).set_index('Z')

In [10]:
import pylake

/home/leroquan@eawag.wroot.emp-eaw.ch/miniconda3/envs/horizontal_structures/lib/python3.11/site-packages/pylake/pylake.py:3: UserWarning: The seawater library is deprecated! Please use gsw instead.
  import seawater as sw


In [11]:
date_list_strat = pd.to_datetime(stratification.columns)
thermocline_depths_per_date = []
for idx_date, mode_date in enumerate(stratification.columns):
    rho = pylake.dens0(s=0.2, t=stratification[mode_date].values[:-1])
    thermocline_depth, thermocline_idx = pylake.thermocline(stratification[mode_date].values[:-1], -1*stratification.index.values[:-1])
    thermocline_depths_per_date.append(thermocline_depth)

In [12]:
total_depths_u = ds_mitgcm.drF.expand_dims(XG=ds_mitgcm.XG, YC=ds_mitgcm.YC).where(ds_mitgcm.isel(time=-1).UVEL > 0).sum(dim='Z').drop_vars('time').load()
total_depths_v = ds_mitgcm.drF.expand_dims(XC=ds_mitgcm.XC, YG=ds_mitgcm.YG).where(ds_mitgcm.isel(time=-1).VVEL > 0).sum(dim='Z').drop_vars('time').load()

In [13]:
ds_mitgcm = ds_mitgcm.chunk({'time':-1, 'Z':-1, 'XG':10, 'XC':10, 'YG':10, 'YC':10 })

In [14]:
date_list_strat[:8]

DatetimeIndex(['2025-04-07', '2025-04-22', '2025-05-07', '2025-05-22',
               '2025-06-07', '2025-06-22', '2025-07-07', '2025-07-22'],
              dtype='datetime64[ns]', freq=None)

# Process each layer

In [15]:
def project_vector_mode_test(
    U,
    V,
    dA,
    mode_u,
    mode_v,
    rho=1025.0,
    normalize_mode=True,
):
    """
    KE-weighted projection of MITgcm vector velocity fields onto
    a possibly complex vector mode.

    Parameters
    ----------
    U, V : xarray.DataArray
        Velocity fields on MITgcm C-grid:
            U : (time, Z, YC, XG)
            V : (time, Z, YG, XC)

    dA : xarray.DataArray
        Horizontal cell area.

    mode_u, mode_v : xarray.DataArray
        Complex modal structure on the same grids as U and V.

    rho : float
        Reference density.

    normalize_mode : bool
        If True, normalize mode so that ||mode||² = 1
        under the KE inner product.

    Returns
    -------
    A_da : xr.DataArray
        Complex modal amplitude A(t)

    KE_da : xr.DataArray
        Modal kinetic energy:
            KE = 0.5 * |A|²
        if normalize_mode=True

    U_proj_phys, V_proj_phys : xr.DataArray
        Physical reconstructed velocities:
            Re(A * mode)

    U_proj_complex, V_proj_complex : xr.DataArray
        Full complex reconstructed fields.
    """

    # ------------------------------------------------------------
    # Match mode grids to velocity grids
    # ------------------------------------------------------------

    mode_u = _match_mode_to_field(U, mode_u)
    mode_v = _match_mode_to_field(V, mode_v)

    has_z = "Z" in U.dims
    if has_z:
        rho_u_arr = xr.DataArray(
            np.full(U["drFu"].size, rho),
            coords={"Z": U["drFu"]["Z"]},
            dims=("Z",),
        )
        rho_v_arr = xr.DataArray(
            np.full(V["drFv"].size, rho),
            coords={"Z": V["drFv"]["Z"]},
            dims=("Z",),
        )
    else:
        rho_u_arr = rho
        rho_v_arr = rho

    # ------------------------------------------------------------
    # KE weighting
    #
    # Inner product:
    #
    # <u,v> = ∫ rho * u * v* dV
    #
    # We implement this via sqrt(weights)
    # ------------------------------------------------------------

    w_u = np.sqrt(rho_u_arr * U["drFu"] * dA)
    w_v = np.sqrt(rho_v_arr * V["drFv"] * dA)

    # ------------------------------------------------------------
    # Weighted fields
    # ------------------------------------------------------------

    if has_z:
        u_space = ("Z", "YC", "XG")
        v_space = ("Z", "YG", "XC")
    else:
        u_space = ("YC", "XG")
        v_space = ("YG", "XC")

    Uw = (U * w_u).stack(space=u_space)
    Vw = (V * w_v).stack(space=v_space)

    mode_uw = (mode_u * w_u).stack(space=u_space)
    mode_vw = (mode_v * w_v).stack(space=v_space)

    Uw_np = Uw.values
    Vw_np = Vw.values

    mode_uw_np = mode_uw.values
    mode_vw_np = mode_vw.values

    # ------------------------------------------------------------
    # Mode norm
    # ------------------------------------------------------------

    norm2 = (
        np.vdot(mode_uw_np, mode_uw_np).real
        + np.vdot(mode_vw_np, mode_vw_np).real
    )

    if normalize_mode:
        norm = np.sqrt(norm2)

        mode_u = mode_u / norm
        mode_v = mode_v / norm

        mode_uw_np = mode_uw_np / norm
        mode_vw_np = mode_vw_np / norm

        norm2 = 1.0

    # ------------------------------------------------------------
    # Modal amplitude
    #
    # A(t) = <u, mode>
    # ------------------------------------------------------------

    A = (
        Uw_np @ mode_uw_np.conj()
        + Vw_np @ mode_vw_np.conj()
    )

    # ------------------------------------------------------------
    # Modal KE
    #
    # If normalized:
    #     KE = 0.5 |A|²
    # ------------------------------------------------------------

    KE = 0.5 * np.abs(A) ** 2 / norm2

    # ------------------------------------------------------------
    # Complex reconstruction
    #
    # u_proj = A * mode / ||mode||²
    # ------------------------------------------------------------

    A_expanded = A[:, None]

    mode_u_stack = mode_u.stack(space=u_space).values
    mode_v_stack = mode_v.stack(space=v_space).values

    U_proj_complex_np = (
        A_expanded * mode_u_stack / norm2
    )

    V_proj_complex_np = (
        A_expanded * mode_v_stack / norm2
    )

    # ------------------------------------------------------------
    # Convert back to xarray
    # ------------------------------------------------------------

    U_proj_complex = xr.DataArray(
        U_proj_complex_np.reshape(Uw.shape),
        coords=Uw.coords,
        dims=Uw.dims,
    ).unstack("space")

    V_proj_complex = xr.DataArray(
        V_proj_complex_np.reshape(Vw.shape),
        coords=Vw.coords,
        dims=Vw.dims,
    ).unstack("space")


    # ------------------------------------------------------------
    # Output DataArrays
    # ------------------------------------------------------------

    A_da = xr.DataArray(
        A,
        coords={"time": U["time"]},
        dims=("time",),
        name="A_mode",
    )

    KE_da = xr.DataArray(
        KE,
        coords={"time": U["time"]},
        dims=("time",),
        name="KE_mode",
    )

    return {
        'A_real': A_da.real,
        'A_imag': A_da.imag,
        'KE': KE_da,
        'U_real': U_proj_complex.real,
        'V_real': V_proj_complex.real,
        'U_imag': U_proj_complex.imag,
        'V_imag': V_proj_complex.imag,
        'h': U["drFu"].where(U!=0).broadcast_like(U)
    }

In [16]:
# ------------------------------------------------------------
# Load modes once
# ------------------------------------------------------------

mode_cache = []

for nc_file in files:

    ds_mode = xr.open_dataset(nc_file).load()

    ds_mode["u1"] = (
        ds_mode.u1_real +
        1j * ds_mode.u1_imag
    ).fillna(0)

    ds_mode["v1"] = (
        ds_mode.v1_real +
        1j * ds_mode.v1_imag
    ).fillna(0)

    ds_mode["u2"] = (
        ds_mode.u2_real +
        1j * ds_mode.u2_imag
    ).fillna(0)

    ds_mode["v2"] = (
        ds_mode.v2_real +
        1j * ds_mode.v2_imag
    ).fillna(0)

    mode_cache.append(
        (
            str(nc_file),
            ds_mode
        )
    )

In [17]:
def process_mode(
    mode_name,
    ds_mode,
    U,
    V,
):

    mode_u = (
        ds_mode.u1,
        ds_mode.u2,
    )

    mode_v = (
        ds_mode.v1,
        ds_mode.v2,
    )


    results = []

    for layer in (0,1):

        result = project_vector_mode_test(
            U[layer],
            V[layer],
            horizontal_resolution**2,
            mode_u[layer],
            mode_v[layer],
            rho=1025.0,
        )

        results.append(result)


    return xr.Dataset(
        {
            "KE":
                xr.concat(
                    [
                        r["KE"].expand_dims(layer=[i])
                        for i,r in enumerate(results)
                    ],
                    dim="layer",
                ),

            "A_real":
                xr.concat(
                    [
                        r["A_real"].expand_dims(layer=[i])
                        for i,r in enumerate(results)
                    ],
                    dim="layer",
                ),

            "A_imag":
                xr.concat(
                    [
                        r["A_imag"].expand_dims(layer=[i])
                        for i,r in enumerate(results)
                    ],
                    dim="layer",
                ),

            "U_real":
                xr.concat(
                    [
                        r["U_real"].expand_dims(layer=[i])
                        for i,r in enumerate(results)
                    ],
                    dim="layer",
                ),

            "U_imag":
                xr.concat(
                    [
                        r["U_imag"].expand_dims(layer=[i])
                        for i,r in enumerate(results)
                    ],
                    dim="layer",
                ),

            "V_real":
                xr.concat(
                    [
                        r["V_real"].expand_dims(layer=[i])
                        for i,r in enumerate(results)
                    ],
                    dim="layer",
                ),

            "V_imag":
                xr.concat(
                    [
                        r["V_imag"].expand_dims(layer=[i])
                        for i,r in enumerate(results)
                    ],
                    dim="layer",
                ),
            "h":
                xr.concat(
                    [
                        r["h"].expand_dims(layer=[i])
                        for i,r in enumerate(results)
                    ],
                    dim="layer",
                ),

        }
    ).expand_dims(
        mode=[mode_name]
    )

In [18]:
from joblib import Parallel, delayed


ds_all = []

for idx_strat_date, strat_date in enumerate(date_list_strat[8:]):

    print(strat_date)


    ds_crop = ds_mitgcm.sel(
        time=slice(
            strat_date-pd.Timedelta(days=7),
            strat_date+pd.Timedelta(days=7),
        )
    )


    UVEL = ds_crop.UVEL.load()
    VVEL = ds_crop.VVEL.load()


    h1 = thermocline_depths_per_date[idx_strat_date]


    zmin = float(ds_crop.Z.min())

    U = [
        UVEL.sel(Z=slice(0, -h1))
            .mean(dim="Z")
            .fillna(0),

        UVEL.sel(Z=slice(-h1, zmin))
            .mean(dim="Z")
            .fillna(0),
    ]


    V = [
        VVEL.sel(Z=slice(0, -h1))
            .mean(dim="Z")
            .fillna(0),

        VVEL.sel(Z=slice(-h1, zmin))
            .mean(dim="Z")
            .fillna(0),
    ]


    U[0]["drFu"] = np.minimum(h1, total_depths_u)
    V[0]["drFv"] = np.minimum(h1, total_depths_v)

    U[1]["drFu"] = np.maximum(0, total_depths_u-h1)
    V[1]["drFv"] = np.maximum(0, total_depths_v-h1)



    results = Parallel(
        n_jobs=-1
    )(
        delayed(process_mode)(
            name,
            ds_mode,
            U,
            V,
        )
        for name,ds_mode in mode_cache
    )


    ds_all.append(
        xr.concat(
            results,
            dim="mode"
        )
        .expand_dims(
            strat_date=[strat_date]
        )
    )


ds_all = xr.concat(
    ds_all,
    dim="strat_date"
)

2025-08-07 00:00:00


/tmp/ipykernel_220909/4136035530.py:65: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
/tmp/ipykernel_220909/4136035530.py:74: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
/tmp/ipykernel_220909/4136035530.py:83: FutureWarning: In a future version of xarray the default value for coords will change from coords='differe

2025-08-22 00:00:00


/tmp/ipykernel_220909/4136035530.py:65: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
/tmp/ipykernel_220909/4136035530.py:74: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
/tmp/ipykernel_220909/4136035530.py:83: FutureWarning: In a future version of xarray the default value for coords will change from coords='differe

2025-09-07 00:00:00


...

/tmp/ipykernel_220909/4136035530.py:65: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
/tmp/ipykernel_220909/4136035530.py:74: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
/tmp/ipykernel_220909/4136035530.py:83: FutureWarning: In a future version of xarray the default value for coords will change from coords='differe

2025-11-07 00:00:00


/tmp/ipykernel_220909/4136035530.py:65: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
/tmp/ipykernel_220909/4136035530.py:74: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
/tmp/ipykernel_220909/4136035530.py:83: FutureWarning: In a future version of xarray the default value for coords will change from coords='differe

2025-11-22 00:00:00


/tmp/ipykernel_220909/4136035530.py:65: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
/tmp/ipykernel_220909/4136035530.py:74: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
/tmp/ipykernel_220909/4136035530.py:83: FutureWarning: In a future version of xarray the default value for coords will change from coords='differe

In [21]:
ds_all.sum(dim='strat_date').to_netcdf(os.path.join(modal_analysis_dir, f"KE_projected_2layers_02.nc"))

In [20]:
ds_all

<xarray.Dataset> Size: 11GB
Dimensions:     (strat_date: 8, mode: 1, layer: 2, time: 2688, YC: 140, XG: 50,
                 YG: 140, XC: 50)
Coordinates: (12/19)
  * mode        (mode) object 8B '/storage/alplakes_test/zug_100m_2025/modal_...
  * layer       (layer) int64 16B 0 1
  * time        (time) datetime64[ns] 22kB 2025-07-31T00:30:00 ... 2025-11-28...
  * YC          (YC) float64 1kB 50.0 150.0 250.0 ... 1.385e+04 1.395e+04
  * XG          (XG) int64 400B 0 100 200 300 400 ... 4500 4600 4700 4800 4900
  * YG          (YG) int64 1kB 0 100 200 300 400 ... 13600 13700 13800 13900
    ...          ...
    drFu        (strat_date, layer, YC, XG) float32 448kB 0.0 0.0 ... 0.0 0.0
    dxG         (YG, XC) >f4 28kB 0.0 0.0 0.0 0.0 ... 100.0 100.0 100.0 100.0
    dyC         (YG, XC) >f4 28kB 0.0 0.0 0.0 0.0 ... 100.0 100.0 100.0 100.0
    rAs         (YG, XC) >f4 28kB 0.0 0.0 0.0 0.0 ... 1e+04 1e+04 1e+04 1e+04
    maskInS     (YG, XC) bool 7kB False False False False ... False False False
    drFv        (strat_date, layer, YG, XC) float32 448kB 0.0 0.0 ... 0.0 0.0
Data variables:
    KE          (strat_date, mode, layer, time) float64 344kB 1.594e+07 ... 8...
    A_real      (strat_date, mode, layer, time) float64 344kB 5.623e+03 ... -...
    A_imag      (strat_date, mode, layer, time) float64 344kB 517.5 ... 73.56
    U_real      (strat_date, mode, layer, time, YC, XG) float64 2GB 0.0 ... 0.0
    U_imag      (strat_date, mode, layer, time, YC, XG) float64 2GB 0.0 ... 0.0
    V_real      (strat_date, mode, layer, time, YG, XC) float64 2GB 0.0 ... 0.0
    V_imag      (strat_date, mode, layer, time, YG, XC) float64 2GB 0.0 ... 0.0
    h           (strat_date, mode, layer, time, YC, XG) float32 1GB nan ... nan